In [1]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import DataLoader, Dataset

In [2]:
DATASET_PATH = Path("../neural_fx_dataset/")
TEST_DI = DATASET_PATH / "DI.wav"
TEST_EFFECT = DATASET_PATH / "tsmini_gain_100.wav"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
class _L(nn.LSTM):
    """
    Tweaks to PyTorch LSTM module
    * Up the remembering
    """

    def reset_parameters(self) -> None:
        super().reset_parameters()
        # https://danijar.com/tips-for-training-recurrent-neural-networks/
        # forget += 1
        # ifgo
        value = 2.0
        idx_input = slice(0, self.hidden_size)
        idx_forget = slice(self.hidden_size, 2 * self.hidden_size)
        for layer in range(self.num_layers):
            for input in ("i", "h"):
                # Balance out the scale of the cell w/ a -=1
                getattr(self, f"bias_{input}h_l{layer}").data[idx_input] -= value
                getattr(self, f"bias_{input}h_l{layer}").data[idx_forget] += value

class NeuralfxLSTM(nn.Module):
    def __init__(self, input_size, conv1d_filters, conv1d_strides, hidden_units):
        super(NeuralfxLSTM, self).__init__()
        
        padding = (12 - 1) // 2
        
        self.conv1 = nn.Conv1d(in_channels=1, 
                               out_channels=conv1d_filters, 
                               kernel_size=12, 
                               stride=conv1d_strides, 
                               padding=padding)
        
        self.conv2 = nn.Conv1d(in_channels=conv1d_filters, 
                               out_channels=conv1d_filters, 
                               kernel_size=12, 
                               stride=conv1d_strides, 
                               padding=padding)
        
        self.lstm = nn.LSTM(input_size=conv1d_filters, 
                            hidden_size=hidden_units, 
                            batch_first=True)
        
        self.fc = nn.Linear(hidden_units, 1)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        
        # Reshape for LSTM (batch_size, seq_length, features)
        x = x.transpose(1, 2)
        
        x, _ = self.lstm(x)
        
        # Take the last output of the LSTM
        x = x[:, -1, :]
        
        x = self.fc(x)
        return x

In [4]:
def pre_emphasis_filter(x, coeff=0.95):
    return torch.concat([x, x - coeff * x], 1)


def ESR(y_pred, y_true):
    """
    Error to signal ratio with pre-emphasis filter:
    """
    y_true, y_pred = pre_emphasis_filter(y_true), pre_emphasis_filter(y_pred)
    return torch.sum(torch.pow(y_true - y_pred, 2)) / torch.sum(torch.pow(y_true, 2)) + 1e-10

In [5]:
x_waveform, x_samplerate = torchaudio.load(TEST_DI)
y_waveform, y_samplerate = torchaudio.load(TEST_EFFECT)

assert(x_samplerate == y_samplerate)

class AudioDataset(Dataset):
    def __init__(self, file_paths, chunk_size=2048, overlap=0):
        self.file_paths = file_paths
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.data = []

        self._process_files()

    def _process_files(self):
        for x_path, y_path in self.file_paths:
            x, _ = torchaudio.load(x_path)
            y, _ = torchaudio.load(y_path)
            
            # Ensure the output matches the input length
            y = y[:, :x.shape[1]]
            
            # Chunk the audio
            step = self.chunk_size - self.overlap
            for i in range(0, x.shape[1] - self.chunk_size + 1, step):
                x_chunk = x[:, i:i + self.chunk_size]
                y_chunk = y[:, i:i + self.chunk_size]
                self.data.append((x_chunk, y_chunk))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [6]:
def train_model(model, dataloader, criterion, optimizer, num_epochs=10):
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, (inputs, targets) in enumerate(dataloader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets.squeeze())
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(dataloader)
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

In [7]:
# Hyperparameters and model initialization
learning_rate = 0.01
conv1d_filters = 16
conv1d_strides = 12
hidden_units = 36
input_size = 2048
chunk_size = 2048
overlap = 1024
batch_size = 16
num_epochs = 10

In [8]:
model = NeuralfxLSTM(input_size, conv1d_filters, conv1d_strides, hidden_units)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Prepare the data
# Replace with actual file paths
file_paths = [(TEST_DI, TEST_EFFECT)]

dataset = AudioDataset(file_paths, chunk_size=chunk_size, overlap=overlap)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Train the model
model.to(device)
train_model(model, dataloader, criterion, optimizer, num_epochs)

/home/ikkjo/anaconda3/envs/neural-fx/lib/python3.11/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([16, 2048])) that is different to the input size (torch.Size([16, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/ikkjo/anaconda3/envs/neural-fx/lib/python3.11/site-packages/torch/nn/modules/loss.py:535: UserWarning: Using a target size (torch.Size([6, 2048])) that is different to the input size (torch.Size([6, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 1/10, Loss: 0.0083
Epoch 2/10, Loss: 0.0082
Epoch 3/10, Loss: 0.0082
Epoch 4/10, Loss: 0.0084
Epoch 5/10, Loss: 0.0082
Epoch 6/10, Loss: 0.0082
Epoch 7/10, Loss: 0.0083
Epoch 8/10, Loss: 0.0083
Epoch 9/10, Loss: 0.0083
Epoch 10/10, Loss: 0.0082
